In [1]:
import pandas as pd

# Data Acquisition

In [2]:
path = "../data/Resume.csv"

resume_df = pd.read_csv(path)

resume_df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [3]:
resume_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2484 entries, 0 to 2483
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ID           2484 non-null   int64 
 1   Resume_str   2484 non-null   object
 2   Resume_html  2484 non-null   object
 3   Category     2484 non-null   object
dtypes: int64(1), object(3)
memory usage: 77.8+ KB


In [4]:
path = "../data/training_data.csv"

job_df = pd.read_csv(path)

job_df.head()

,company_name,job_description,position_title,description_length,model_response
0,Google,minimum qualifications\nbachelors degree or eq...,Sales Specialist,2727,"{\n ""Core Responsibilities"": ""Responsible fo..."
1,Apple,description\nas an asc you will be highly infl...,Apple Solutions Consultant,828,"{\n ""Core Responsibilities"": ""as an asc you ..."
2,Netflix,its an amazing time to be joining netflix as w...,Licensing Coordinator - Consumer Products,3205,"{\n ""Core Responsibilities"": ""Help drive bus..."
3,Robert Half,description\n\nweb designers looking to expand...,Web Designer,2489,"{\n ""Core Responsibilities"": ""Designing webs..."
4,TrackFive,at trackfive weve got big goals were on a miss...,Web Developer,3167,"{\n ""Core Responsibilities"": ""Build and layo..."


In [5]:
job_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 853 entries, 0 to 852
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   company_name        853 non-null    object
 1   job_description     853 non-null    object
 2   position_title      853 non-null    object
 3   description_length  853 non-null    int64 
 4   model_response      853 non-null    object
dtypes: int64(1), object(4)
memory usage: 33.4+ KB


# Data Cleaning

### Missing Values

In [15]:
cols_with_question = resume_df.columns[resume_df.isin(['?']).any()].tolist()
print(cols_with_question)

[]


In [12]:
cols_with_question = job_df.columns[job_df.isin(['?']).any()].tolist()
print(cols_with_question)

[]


In [14]:
print( (job_df.isna().sum() / len(job_df) ) * 100)

company_name          0.0
job_description       0.0
position_title        0.0
description_length    0.0
model_response        0.0
clean_description     0.0
dtype: float64


In [16]:
print( (resume_df.isna().sum() / len(resume_df) ) * 100)

ID              0.0
Resume_str      0.0
Resume_html     0.0
Category        0.0
clean_resume    0.0
dtype: float64


### Duplicates

In [17]:
job_df.duplicated().sum()

np.int64(0)

In [18]:
resume_df.duplicated().sum()

np.int64(0)

## Data Preprocessing

### Text Cleaning
- remove html
- remove whitespace
- remove extra chars
- lowercase

In [6]:
from bs4 import BeautifulSoup
import re

def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text)
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9+#.\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [7]:
job_df["clean_description"] = job_df["job_description"].apply(clean_text)
resume_df["clean_resume"] = resume_df["Resume_str"].apply(clean_text)

In [8]:
print(job_df["clean_description"], '\n')
print(resume_df["clean_resume"])

0      minimum qualifications bachelors degree or equ...
1      description as an asc you will be highly influ...
2      its an amazing time to be joining netflix as w...
3      description web designers looking to expand yo...
4      at trackfive weve got big goals were on a miss...
                             ...                        
848    job description parttime make big money at men...
849    responsibilities parkers internship program wa...
850    the borgen project is an innovative national c...
851    put the world on vacation at wyndham destinati...
852    this job handles customer inquiries by telepho...
Name: clean_description, Length: 853, dtype: object 

0       hr administrator marketing associate hr admini...
1       hr specialist us hr operations summary versati...
2       hr director summary over 20 years experience i...
3       hr specialist summary dedicated driven and dyn...
4       hr manager skill highlights hr skills hr depar...
                             

In [9]:
job_df["clean_description"][1]

'description as an asc you will be highly influential in growing mind and market share of apple products while building longterm relationships with those who share your passion customer experiences are driven through you and your partner team growing in an ever changing and challenging environment you strive for perfection whether its maintaining visual merchandising or helping to grow and develop your partner team qualifications a passion to help people understand how apple products can enrich their livesexcellent communication skills allowing you to be as comfortable in front of a small group as you are speaking with individuals years preferred working in a dynamic sales andor results driven environment as well as proven success developing customer loyaltyability to encourage a partner team and grow apple business'

### Keyword Extraction
- skills
- responsibilites
- qualifications
- requirements
- degree
- experience